In [1]:
import torch
import pandas as pd
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    classification_report
)

data = pd.read_csv("../data/synthetic_data.csv")

print(data.head())
print(data.info())
print(data.describe())
print(data.shape)

class IrisNeuralNetwork:
    def __init__(self, input_size, hidden1_size, hidden2_size):

        # Input -> hidden layer 1
        self.w12 = torch.randn(input_size, hidden1_size)
        self.b1 = torch.zeros(hidden1_size)

        # Hidden layer 1 -> hidden layer 2
        self.w23 = torch.randn(hidden1_size, hidden2_size)
        self.b2 = torch.zeros(hidden2_size)

        # Hidden layer 2 -> output
        self.w34 = torch.randn(hidden2_size, 1)
        self.b3 = torch.zeros(1)

    def sigmoid(self, x):
        return 1 / (1 + torch.exp(-x))

    def forward(self, x):
        # Layer 1
        z1 = x @ self.w12 + self.b1
        a1 = self.sigmoid(z1)

        # Layer 2
        z2 = a1 @ self.w23 + self.b2
        a2 = self.sigmoid(z2)

        # Output layer
        z3 = a2 @ self.w34 + self.b3
        a3 = self.sigmoid(z3)

        return a3

history = {
    'train_loss': [],
    'test_loss': [],
    'test_accuracy': []
}

         X1        X2  output
0  0.994416  0.533942       1
1  1.177820 -1.126290       0
2 -0.945938 -0.848435       0
3  0.936008 -0.305729       1
4 -0.694159  0.698876       1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      1000 non-null   float64
 1   X2      1000 non-null   float64
 2   output  1000 non-null   int64  
dtypes: float64(2), int64(1)
memory usage: 23.6 KB
None
                X1           X2       output
count  1000.000000  1000.000000  1000.000000
mean     -0.027157    -0.041622     0.489000
std       0.881587     0.867416     0.500129
min      -1.499560    -1.493940     0.000000
25%      -0.776087    -0.756851     0.000000
50%      -0.031119    -0.085191     0.000000
75%       0.767149     0.699188     1.000000
max       1.496600     1.499670     1.000000
(1000, 3)


In [2]:
x = torch.tensor(
    data[["X1", "X2"]].values,
    dtype=torch.float32
)

y = torch.tensor(
    data["output"].values,
    dtype=torch.float32
).reshape(-1, 1)

print(x.shape, y.shape)

N = int(0.8 * len(x))

x_train, x_test = x[:N], x[N:]
y_train, y_test = y[:N], y[N:]

print(f"Training set: {x_train.shape}, {y_train.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")

torch.Size([1000, 2]) torch.Size([1000, 1])
Training set: torch.Size([800, 2]), torch.Size([800, 1])
Test set: torch.Size([200, 2]), torch.Size([200, 1])


In [3]:
def binary_cross_entropy(y, prediction):

    # Prevent log(0)
    epsilon = 1e-7

    prediction = torch.clamp(
        prediction,
        epsilon,
        1 - epsilon
    )

    loss = -(
        y * torch.log(prediction)
        +
        (1 - y) * torch.log(1 - prediction)
    )

    return loss.mean()

def backward(model, x, y, prediction):

    # Number of samples
    m = x.shape[0]

    # -----------------------------------------
    # Recalculate forward-pass intermediates
    # -----------------------------------------

    z1 = x @ model.w12 + model.b1
    a1 = model.sigmoid(z1)

    z2 = a1 @ model.w23 + model.b2
    a2 = model.sigmoid(z2)

    z3 = a2 @ model.w34 + model.b3
    a3 = prediction

    # -----------------------------------------
    # Output layer
    # -----------------------------------------

    # For sigmoid + binary cross entropy:
    # dL/dz3 = a3 - y
    dz3 = a3 - y

    dw34 = (a2.T @ dz3) / m
    db3 = dz3.mean(dim=0)

    # -----------------------------------------
    # Hidden layer 2
    # -----------------------------------------

    da2 = dz3 @ model.w34.T

    dz2 = da2 * a2 * (1 - a2)

    dw23 = (a1.T @ dz2) / m
    db2 = dz2.mean(dim=0)

    # -----------------------------------------
    # Hidden layer 1
    # -----------------------------------------

    da1 = dz2 @ model.w23.T

    dz1 = da1 * a1 * (1 - a1)

    dw12 = (x.T @ dz1) / m
    db1 = dz1.mean(dim=0)

    return dw12, db1, dw23, db2, dw34, db3


def update_parameters(model, gradients, learning_rate, count):

    dw12, db1, dw23, db2, dw34, db3 = gradients

    model.w12 -= learning_rate * dw12
    model.b1  -= learning_rate * db1

    model.w23 -= learning_rate * dw23
    model.b2  -= learning_rate * db2

    model.w34 -= learning_rate * dw34
    model.b3  -= learning_rate * db3

    # if count % 100 == 0:
    #     print("Updated parameters:\n")
    #     print(f"  W12: {model.w12.shape}")
    #     print(f"  B1: {model.b1.shape}\n")
    #     print(f"  W23: {model.w23.shape}")
    #     print(f"  B2: {model.b2.shape}\n")
    #     print(f"  W34: {model.w34.shape}")
    #     print(f"  B3: {model.b3.shape}\n")


In [5]:
def train(model, x_train, y_train, epochs, learning_rate, count):

    for epoch in range(epochs):
        count+=1    

        # Forward pass
        prediction = model.forward(x_train)

        # Calculate loss
        train_loss = binary_cross_entropy(
            y_train,
            prediction
        )

        # Backpropagation
        gradients = backward(
            model,
            x_train,
            y_train,
            prediction
        )

        # dw12, db1, dw23, db2, dw34, db3 = gradients

        # if epoch % 100 == 0:
        #     print(
        #         f"dw12={dw12.abs().mean().item():.8f}, "
        #         f"dw23={dw23.abs().mean().item():.8f}, "
        #         f"dw34={dw34.abs().mean().item():.8f}"
        #     )

        # Gradient descent
        update_parameters(
            model,
            gradients,
            learning_rate,
            count
        )

        history['train_loss'].append(train_loss.item())        

        if epoch % 100 == 0:
            print(
                f"Epoch {epoch:4d} | "
                f"Loss: {train_loss.item():.6f}"
            )


In [6]:
model = torch.load(
    "../../Iris_neural_net/models/iris_nn_model.pt",
    weights_only=False
)

train(
    model,
    x_train,
    y_train,
    epochs=5000,
    learning_rate=0.1,
    count=0
)

Epoch    0 | Loss: 2.293056
Epoch  100 | Loss: 0.693424
Epoch  200 | Loss: 0.693278
Epoch  300 | Loss: 0.693168
Epoch  400 | Loss: 0.693073
Epoch  500 | Loss: 0.692990
Epoch  600 | Loss: 0.692918
Epoch  700 | Loss: 0.692853
Epoch  800 | Loss: 0.692795
Epoch  900 | Loss: 0.692743
Epoch 1000 | Loss: 0.692695
Epoch 1100 | Loss: 0.692650
Epoch 1200 | Loss: 0.692608
Epoch 1300 | Loss: 0.692567
Epoch 1400 | Loss: 0.692529
Epoch 1500 | Loss: 0.692491
Epoch 1600 | Loss: 0.692454
Epoch 1700 | Loss: 0.692417
Epoch 1800 | Loss: 0.692379
Epoch 1900 | Loss: 0.692341
Epoch 2000 | Loss: 0.692302
Epoch 2100 | Loss: 0.692261
Epoch 2200 | Loss: 0.692219
Epoch 2300 | Loss: 0.692174
Epoch 2400 | Loss: 0.692126
Epoch 2500 | Loss: 0.692075
Epoch 2600 | Loss: 0.692021
Epoch 2700 | Loss: 0.691962
Epoch 2800 | Loss: 0.691898
Epoch 2900 | Loss: 0.691829
Epoch 3000 | Loss: 0.691753
Epoch 3100 | Loss: 0.691670
Epoch 3200 | Loss: 0.691579
Epoch 3300 | Loss: 0.691479
Epoch 3400 | Loss: 0.691368
Epoch 3500 | Loss: 0

In [7]:
# -----------------------------------------
# Test
# -----------------------------------------

def evaluate(model, x_test, y_test):
    with torch.no_grad():

        probabilities = model.forward(x_test)

        loss = binary_cross_entropy(
            y_test,
            probabilities)

        predictions = (probabilities >= 0.5).float()

        accuracy = (
            predictions == y_test
        ).float().mean()

        print()
        print("Test results:")
        print(
            f"Overall score: {accuracy.item():.7f} | ",
            f"Test Loss: {loss:.6f}"
            )

        print()

        for i in range(len(y_test)):

            actual = int(y_test[i].item())
            predicted = int(predictions[i].item())
            probability = probabilities[i].item()

            print(
                f"{i:03d}: "
                f"actual: {actual} "
                f"predict: {predicted} "
                f"prob: {probability:.7f}"
            )

        return loss.item(), accuracy.item(), predictions, probabilities


test_loss, accuracy, predictions, probabilities = evaluate(
    model, 
    x_test,
    y_test
)

history['test_loss'].append(test_loss)
history['test_accuracy'].append(accuracy)



Test results:
Overall score: 0.4550000 |  Test Loss: 0.689089

000: actual: 0 predict: 1 prob: 0.5109000
001: actual: 1 predict: 0 prob: 0.4896474
002: actual: 0 predict: 1 prob: 0.5227725
003: actual: 1 predict: 1 prob: 0.5150584
004: actual: 1 predict: 0 prob: 0.4987021
005: actual: 0 predict: 0 prob: 0.4317366
006: actual: 0 predict: 0 prob: 0.4543670
007: actual: 0 predict: 0 prob: 0.4671711
008: actual: 1 predict: 1 prob: 0.5176334
009: actual: 0 predict: 0 prob: 0.4838808
010: actual: 0 predict: 1 prob: 0.5089531
011: actual: 0 predict: 1 prob: 0.5127449
012: actual: 1 predict: 0 prob: 0.4820647
013: actual: 0 predict: 1 prob: 0.5126416
014: actual: 1 predict: 1 prob: 0.5063418
015: actual: 1 predict: 1 prob: 0.5170454
016: actual: 1 predict: 0 prob: 0.4947457
017: actual: 1 predict: 0 prob: 0.4922971
018: actual: 0 predict: 0 prob: 0.4732460
019: actual: 1 predict: 1 prob: 0.5150137
020: actual: 1 predict: 0 prob: 0.4745601
021: actual: 1 predict: 0 prob: 0.4896073
022: actual:

In [8]:
accuracy = accuracy_score(
    y_test.numpy(),
    predictions.numpy()
)

precision = precision_score(
    y_test.numpy(),
    predictions.numpy()
)

recall = recall_score(
    y_test.numpy(),
    predictions.numpy()
)

f1 = f1_score(
    y_test.numpy(),
    predictions.numpy()
)

cm = confusion_matrix(
    y_test.numpy(),
    predictions.numpy()
)

report = classification_report(
    y_test.numpy(),
    predictions.numpy(),
    output_dict=True
)

print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(report)

Confusion Matrix:
[[57 39]
 [70 34]]
Classification Report:
{'0.0': {'precision': 0.44881889763779526, 'recall': 0.59375, 'f1-score': 0.5112107623318386, 'support': 96.0}, '1.0': {'precision': 0.4657534246575342, 'recall': 0.3269230769230769, 'f1-score': 0.384180790960452, 'support': 104.0}, 'accuracy': 0.455, 'macro avg': {'precision': 0.45728616114766474, 'recall': 0.46033653846153844, 'f1-score': 0.4476957766461453, 'support': 200.0}, 'weighted avg': {'precision': 0.4576248516880595, 'recall': 0.455, 'f1-score': 0.4451551772187175, 'support': 200.0}}


In [9]:
experiment = {
    'name': 'Trained Weights Fine-tuned',
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,                 
    'history': history,
    'classification_report': report,
    'confusion_matrix': cm
}

In [10]:
import joblib
joblib.dump(
    experiment,
    "../dictionary/trained_weights_fine_tuned_experiment.pkl"
)

torch.save(model, "../models/trained_weights_fine_tuned.pt")

In [8]:
print("Class 0:", (y_test == 0).sum().item())
print("Class 1:", (y_test == 1).sum().item())

print("\n\n")

print("Predicted 0:", (predictions == 0).sum().item())
print("Predicted 1:", (predictions == 1).sum().item())

Class 0: 96
Class 1: 104



Predicted 0: 127
Predicted 1: 73
